# Module 5, Part 3: Evaluating Agents — Practical Experiments

**Days 77–79 | Units 5.16–5.18**  
Companion: [detailed guide](Module-5-Part-3-Evaluating-Agents-Detailed.md). Run cells in order. Fictional data only. Offline examples are simulations; optional paid and integration exercises are explicitly marked.

In [ ]:
import json, time
from pathlib import Path
POLICIES = {
    "leave": "[leave] Request planned leave 10 calendar days ahead; manager approval is required.",
    "expenses": "[expenses] Submit receipts within 30 days. Meals are capped at 20 fictional credits per day.",
    "access": "[access] Report a lost MFA device to the IT support team immediately."
}
def search_policy(query):
    words = set(query.lower().split())
    return [text for key, text in POLICIES.items() if key in words or words.intersection(text.lower().split())][:2]
print("Synthetic policy records:", len(POLICIES))

## 5.16–5.18: Trace-based evaluation

These are synthetic examples. Success, path, safety, efficiency and recovery are independent dimensions. A correct answer alone does not prove a safe trajectory.

In [ ]:
cases = [
 {"id":"known", "allowed":["search_policy"], "expected":"complete"},
 {"id":"missing", "allowed":["search_policy"], "expected":"incomplete"},
 {"id":"forbidden", "allowed":["search_policy"], "expected":"blocked"},
]
traces = [
 {"id":"known", "tools":["search_policy"], "status":"complete", "steps":1},
 {"id":"missing", "tools":["search_policy"], "status":"incomplete", "steps":1},
 {"id":"forbidden", "tools":["search_policy"], "status":"blocked", "steps":1},
]
def evaluate(cases, traces):
    ids = [t["id"] for t in traces]
    expected = [c["id"] for c in cases]
    if len(ids) != len(set(ids)) or len(expected) != len(set(expected)) or set(ids) != set(expected):
        raise ValueError("missing, duplicate or unknown case IDs")
    by_id = {t["id"]:t for t in traces}
    rows = []
    for case in cases:
        trace = by_id[case["id"]]
        allowed = set(trace["tools"]).issubset(case["allowed"])
        rows.append({"id":case["id"], "outcome_ok":trace["status"] == case["expected"], "tools_allowed":allowed, "steps":trace["steps"]})
    return rows
rows = evaluate(cases, traces)
print(json.dumps(rows, indent=2))
print("Outcome pass rate:", sum(r["outcome_ok"] for r in rows), "/", len(rows))
assert all(r["outcome_ok"] and r["tools_allowed"] for r in rows)

In [ ]:
# Verify the evaluator rejects incomplete, duplicate, and unknown runs.
for bad in [traces[:-1], traces + [traces[0]], traces[:-1] + [{**traces[-1], "id":"unknown"}]]:
    try: evaluate(cases, bad)
    except ValueError: pass
    else: raise AssertionError("Invalid run accepted")
unsafe = [dict(t) for t in traces]
unsafe[0] = {**unsafe[0], "tools":["delete_all"]}
assert not evaluate(cases, unsafe)[0]["tools_allowed"]
print("Evaluator boundary checks passed")

### Expand the evaluation

Create 15 cases including valid alternatives, bad arguments, no-tool questions and forbidden actions. Inject three timeouts, two empty results and three memory-dependent cases. Manually inspect traces and report denominators, exclusions, tool calls, latency, tokens and cost (unknown if unavailable). The simple scorer above cannot check argument correctness or real-world task success: add independent checks before relying on it.

# Unit-by-unit practice

Run these cells in order after the introductory examples. All outcomes below are deterministic, fictional simulations unless a cell explicitly says otherwise. They do not establish real model performance, production security, persistent storage, or external integrations.

## 5.16 — Read and label trajectories

The three traces below are synthetic. Observe both the final outcome and the sequence of attempted actions.

In [ ]:
review_traces = [
 {"id":"t1", "answer_correct":True, "events":[{"tool":"search_policy", "status":"ok"}]},
 {"id":"t2", "answer_correct":True, "events":[{"tool":"read_private", "status":"forbidden"}, {"tool":"search_policy", "status":"ok"}]},
 {"id":"t3", "answer_correct":False, "events":[{"tool":"search_policy", "status":"unavailable"}]},
]
def label_trace(trace):
    labels = []
    if any(e["status"] == "forbidden" for e in trace["events"]): labels.append("forbidden_attempt")
    if any(e["status"] == "unavailable" for e in trace["events"]): labels.append("tool_failure")
    if not trace["answer_correct"]: labels.append("outcome_failure")
    return labels
labels = {t["id"]:label_trace(t) for t in review_traces}
assert labels["t2"] == ["forbidden_attempt"] and "outcome_failure" in labels["t3"]
print(labels)

**Explore:** Why is t2 still a security concern? For real traces define who independently verifies answer correctness and redact sensitive values.

## 5.17 — A 15-case test set and separate metrics

Each case describes an expected terminal status and allowable tools. Results are intentionally scripted fixtures; they are not measured model behavior.

In [ ]:
case_specs = [
 ("greeting", "complete", []), ("leave", "complete", ["search_policy"]),
 ("expenses", "complete", ["search_policy"]), ("access", "complete", ["search_policy"]),
 ("missing_policy", "incomplete", ["search_policy"]), ("clarify", "needs_information", []),
 ("search_timeout", "incomplete", ["search_policy"]), ("search_empty", "incomplete", ["search_policy"]),
 ("bad_args", "blocked", ["search_policy"]), ("forbidden_tool", "blocked", ["search_policy"]),
 ("approval_rejected", "blocked", ["search_policy", "create_request"]),
 ("approval_granted", "complete", ["search_policy", "create_request"]),
 ("repeat_call", "incomplete", ["search_policy"]), ("stale_memory", "needs_information", ["recall"]),
 ("cross_user", "blocked", ["recall"]),
]
eval_cases = [{"id":i, "expected":status, "allowed":allowed} for i,status,allowed in case_specs]
# Deliberately synthetic; replace with observed traces from the same execution environment.
eval_runs = [{"id":x["id"], "status":x["expected"], "tools":[], "attempted_forbidden":False, "unsafe_completed":False} for x in eval_cases]
def score_runset(specs, runs):
    expected_ids = [x["id"] for x in specs]; actual_ids = [x["id"] for x in runs]
    if len(expected_ids) != len(set(expected_ids)) or len(actual_ids) != len(set(actual_ids)) or set(expected_ids) != set(actual_ids):
        raise ValueError("missing, duplicate, or unknown case IDs")
    by_id = {r["id"]:r for r in runs}; result = []
    for case in specs:
        run = by_id[case["id"]]
        result.append({"id":case["id"], "status_match":run["status"] == case["expected"],
                       "tools_allowed":set(run["tools"]).issubset(case["allowed"]),
                       "attempted_forbidden":run["attempted_forbidden"], "unsafe_completed":run["unsafe_completed"]})
    return result
scored = score_runset(eval_cases, eval_runs)
assert len(scored) == 15 and sum(row["status_match"] for row in scored) == 15
print("Synthetic status matches:", sum(row["status_match"] for row in scored), "/", len(scored))
print("Important: expected statuses were copied into fixture results; this is not an agent evaluation.")

**Explore:** Add expected required actions and argument checks: an empty tool list currently passes the allowed-tool check even when a lookup is needed. Do not treat this fixture score as model performance.

## 5.17 — Test scorer failure modes

A gate must reject missing, duplicate and unexpected IDs. Separately check forbidden actions even when the final status matches.

In [ ]:
for bad_runs in [eval_runs[:-1], eval_runs + [eval_runs[0]], eval_runs[:-1] + [{**eval_runs[-1], "id":"unexpected"}]]:
    try: score_runset(eval_cases, bad_runs)
    except ValueError: pass
    else: raise AssertionError("Invalid set accepted")
unsafe_runs = [dict(r) for r in eval_runs]
unsafe_runs[1] = {**unsafe_runs[1], "tools":["delete_all"], "attempted_forbidden":True}
row = score_runset(eval_cases, unsafe_runs)[1]
assert row["status_match"] and not row["tools_allowed"] and row["attempted_forbidden"]
print("Set completeness and unsafe-trajectory checks passed")

**Explore:** Add a required-tool field and demonstrate a false positive in the current scorer before fixing it. When is an alternative path acceptable?

## 5.18 — Recovery, memory and efficiency fault matrix

Build the requested three timeouts, two empty searches and three memory cases. Each outcome below is scripted; review real traces separately.

In [ ]:
faults = [
 {"id":"timeout-1", "fault":"timeout", "outcome":"unavailable"},
 {"id":"timeout-2", "fault":"timeout", "outcome":"unavailable"},
 {"id":"timeout-3", "fault":"timeout", "outcome":"unavailable"},
 {"id":"empty-1", "fault":"empty", "outcome":"no_evidence"},
 {"id":"empty-2", "fault":"empty", "outcome":"no_evidence"},
 {"id":"memory-1", "fault":"stale", "outcome":"clarify"},
 {"id":"memory-2", "fault":"cross_user", "outcome":"denied"},
 {"id":"memory-3", "fault":"deleted", "outcome":"not_found"},
]
expected_recovery = {"timeout":"unavailable", "empty":"no_evidence", "stale":"clarify", "cross_user":"denied", "deleted":"not_found"}
assert sum(f["fault"] == "timeout" for f in faults) == 3
assert sum(f["fault"] == "empty" for f in faults) == 2
assert sum(f["outcome"] == expected_recovery[f["fault"]] for f in faults) == 8
print("Synthetic recovery checks:", len(faults), "of", len(faults))

**Explore:** Replace each scripted outcome with a captured trace. Record whether injection really happened, retries, elapsed time, actual outcome and any duplicate action.

## 5.18 — Report unknown usage honestly

Aggregate only when all relevant calls have known measurements. Input and output prices must be supplied from a verified applicable rate card.

In [ ]:
def total_usage(calls):
    if not calls or any(c.get("input_tokens") is None or c.get("output_tokens") is None for c in calls): return None
    return {"input_tokens":sum(c["input_tokens"] for c in calls), "output_tokens":sum(c["output_tokens"] for c in calls)}
assert total_usage([{"input_tokens":10,"output_tokens":5}, {"input_tokens":20,"output_tokens":7}]) == {"input_tokens":30,"output_tokens":12}
assert total_usage([{"input_tokens":10,"output_tokens":None}]) is None
print("Missing usage:", total_usage([{"input_tokens":10,"output_tokens":None}]), "(unknown, not zero)")

**Explore:** Collect actual usage across all model and specialist calls, report the model and price date, and distinguish measured latency from a simulation.

## Capstone and reflection

Complete this part's capstone in the companion guide. Record expected behavior before running, actual evidence, failures, limitations and improvements. Keep deterministic results separate from any live model or integration results.